# G03: Analysis

*How much does each graph's evidence help, and does the ordering between graphs survive holding coverage constant?*

| Sub-metric | Definition | Quantification | Scoring |
|---|---|---|---|
| **Grounding gain** | MRR in a KG arm minus MRR in the `none` arm, per model | paired over queries; percentile bootstrap over queries (n from `grounding.n_bootstrap`) | reported per (model, graph) |
| **Graph contrasts** | difference in MRR between two KG arms, per model | paired bootstrap interval; Wilcoxon signed-rank with Benjamini-Hochberg correction within each subset | a contrast separates when its interval excludes zero and survives FDR |
| **Evidence control** | the same contrasts restricted to pairs evaluable in every arm (each graph holds the pair and has evidence for it) | `main` family restricted post hoc, and the `common` family run on that subset by construction | shrinkage under control is the availability share of the apparent difference |
| **Absence penalty** | the lift over `none` when the positive reached the model with no dossier while its distractors carried theirs, against the lift when it was shown | paired bootstrap over pairs, per model and graph, split on `pos_has_evidence` | a negative absent-positive lift is the cost of a coverage gap at deployment |

Reciprocal ranks are averaged to one value per query per arm across seeds and shuffles before anything is bootstrapped, because rows within a query are not independent.

**Reported, not scored.** Parse-failure diagnostics and the miss-corrected MRR; the rank distribution and the evidence-use fields the model reports (`basis`, `evidence_agreement`, `confidence`).

**Inputs:** `results/tables/G02_run/runs/` · `results/checkpoints/{G01_benchmark,G02_run}.pkl` · `config/params.yaml` (`grounding`)

**Outputs:** `results/{figures,tables}/G03_analysis/` · `results/checkpoints/G03_analysis.pkl`

Design notes and justifications: [`README.md`](README.md)

## Set-up

Loads every run file, validates that arms are complete and pair-aligned, and keeps only the headline models (the capacity ladder is G04).

In [ ]:
# Pick up edits to src/ without restarting the kernel. Without this, editing a
# module after it has been imported leaves the kernel holding the old version,
# which surfaces as a confusing ImportError for a name that is plainly on disk.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass                      # not running under IPython (headless runner, plain python)

import gc
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Locate the repo root from any depth, then make the package importable.
# (`pip install -e .` also works and makes this block a no-op.)
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'config' / 'kgs.yaml').exists())
if str(_root / 'src') not in sys.path:
    sys.path.insert(0, str(_root / 'src'))

from IPython.display import display
import numpy as np
import pandas as pd


from biokgsuite.grounding import (absence_penalty_table, arm_table, chance_mrr, contrast_table, diagnostics, load_runs,
                                  significance_tests, validate_runs)
from biokgsuite.viz.grounding import fig_evidence_quality, fig_grounding_gain, fig_rank_distribution
from biokgsuite.io import Outputs, find_config, load_config, require
from biokgsuite.viz.style import setup_style

setup_style()

In [ ]:
# Config, parameters and the results writer.
config = load_config(find_config(_root), trial_subset=False)   # graphs are named below, not by the panel trial file
params = config['analysis_params']
G      = require(params, 'grounding')
BASE   = config['_base_dir']

# Every figure, table and checkpoint goes through `out`, as in notebooks 01-07.
out = Outputs('G03_analysis', BASE)

KGS       = require(G, 'kgs')
MODELS    = require(G, 'models')
LABELS    = require(G, 'model_labels') | {'none': 'No KG', 'primekg': 'PrimeKG', 'drkg': 'DRKG',
                                           'biokg': 'BioKG', 'openbilink': 'OpenBioLink', 'hetionet': 'Hetionet'}
FAMILIES  = require(G, 'families')
POOL_SIZE = require(G, 'pool_size')
SHUFFLES  = require(G, 'shuffles')
SEEDS     = require(G, 'seeds')
STRICT    = require(G, 'strict')
N_BOOT    = require(G, 'n_bootstrap')
SEED      = require(params, 'random_seed')
RNG       = np.random.default_rng(SEED)
# BKG_MODE=mock reads the rehearsal runs 02 wrote under runs_mock/; real
# (the default) reads the results and refuses any mock row it finds there.
MOCK      = os.environ.get('BKG_MODE', 'real') == 'mock'
RUN_DIR   = Path(BASE) / 'results' / 'tables' / 'G02_run' / ('runs_mock' if MOCK else 'runs')

CONTRASTS = [(a, b) for i, a in enumerate(KGS) for b in KGS[i + 1:]]
CHANCE = chance_mrr(POOL_SIZE)
print(f'chance MRR {CHANCE:.3f} at pool size {POOL_SIZE}; contrasts {CONTRASTS}')

### Loading

In [ ]:
bench = out.load_checkpoint('G01_benchmark')['benchmark']
run_ckpt = out.load_checkpoint('G02_run')
runs = validate_runs(load_runs(RUN_DIR, families=FAMILIES, models=MODELS, allow_mock=MOCK))
print(f"mode={run_ckpt['mode']}  {len(runs)} rows  models {sorted(runs.model.unique())}  "
      f"arms {sorted(runs.arm.unique())}  families {sorted(runs.family.unique())}")
main = runs[runs.family == 'main']
common = runs[runs.family == 'common']
COMMON_PAIRS = set(bench.pair_id[bench.common])
main_cov = main[main.pair_id.isin(COMMON_PAIRS)]   # every graph holds the pair and has evidence for it

## 1. Integrity diagnostics

A failed parse scores as a miss (`rank = pool_size + 1`), which costs more in an arm that ranks well, so failures bias against the best arms and make lifts conservative. `mrr_corrected` is what the arm would score if its failures had ranked like its successes.

In [ ]:
diag = diagnostics(runs, POOL_SIZE)
diag['model_label'] = diag.model.map(LABELS).fillna(diag.model)
display(diag.set_index(['family', 'model_label', 'arm'])[['n', 'parse_rate', 'n_error', 'mrr', 'mrr_corrected', 'shift']].round(3))
print('failed calls:', int(diag.n_failed.sum()), 'of', int(diag.n.sum()))
out.table(diag, 'diagnostics')

## 2. Grounding gain and graph contrasts

The `main` family first (every pair), then the `common` family (pairs evaluable in every arm, run as its own experiment).

In [ ]:
ARMS = ['none'] + KGS
arms_main = arm_table(main, 'main', ARMS, N_BOOT, RNG, LABELS)
con_main = contrast_table(main, 'main', CONTRASTS, N_BOOT, RNG, LABELS)
display(arms_main.set_index(['model_label', 'arm_label'])[['n_units', 'mrr', 'lo', 'hi', 'lift', 'lift_lo', 'lift_hi']].round(3))
print('baseline against chance:')
for r in arms_main[arms_main.arm == 'none'].itertuples():
    print(f'  {r.model_label:24s} {r.mrr:.3f}  ({"above" if r.mrr > CHANCE else "below"} chance {CHANCE:.3f})')
display(con_main.set_index(['model_label', 'contrast'])[['n_units', 'diff', 'lo', 'hi', 'separates']].round(3))
out.table(arms_main, 'arms_main'); out.table(con_main, 'contrasts_main')

In [ ]:
if len(common):
    arms_common = arm_table(common, 'common', ARMS, N_BOOT, RNG, LABELS)
    con_common = contrast_table(common, 'common', CONTRASTS, N_BOOT, RNG, LABELS)
    display(arms_common.set_index(['model_label', 'arm_label'])[['n_units', 'mrr', 'lift', 'lift_lo', 'lift_hi']].round(3))
    display(con_common.set_index(['model_label', 'contrast'])[['n_units', 'diff', 'lo', 'hi', 'separates']].round(3))
    out.table(arms_common, 'arms_common'); out.table(con_common, 'contrasts_common')
else:
    arms_common = con_common = None
    print('no common-family runs')

## 3. Evidence control

Two readings of a headline ordering are possible: a graph supplies better evidence, or it supplies evidence more often. Restricting the `main` runs to the pairs every graph can evaluate (holds the pair, has at least one gene or pathway to show for the drug, carries no edge) holds availability constant while leaving pools, seeds and models untouched; the `common` family corroborates it with an independent draw.


In [ ]:
arms_cov = arm_table(main_cov, 'main', ARMS, N_BOOT, RNG, LABELS)
con_cov = contrast_table(main_cov, 'main', CONTRASTS, N_BOOT, RNG, LABELS)
side = (con_main.set_index(['model_label', 'contrast'])[['diff', 'separates']].rename(columns=lambda c: f'{c}_all')
        .join(con_cov.set_index(['model_label', 'contrast'])[['n_units', 'diff', 'separates']].rename(columns=lambda c: f'{c}_covered')))
display(side.round(3))
shrink = (con_main.set_index(['model', 'contrast'])['diff'].abs() - con_cov.set_index(['model', 'contrast'])['diff'].abs())
print(f'contrasts separating from zero: all pairs {int(con_main.separates.sum())}/{len(con_main)}, '
      f'evaluable in every graph {int(con_cov.separates.sum())}/{len(con_cov)}; '
      f'mean shrinkage in |difference| {shrink.mean():+.3f}')
out.table(arms_cov, 'arms_covered'); out.table(con_cov, 'contrasts_covered')

In [ ]:
# Wilcoxon signed-rank rather than a paired t-test: per-query differences are
# bounded and skewed. Benjamini-Hochberg within each subset across its contrasts.
subsets = {'all pairs': main, 'evaluable in every graph': main_cov}
if len(common):
    subsets['common family'] = common
tests = significance_tests(subsets, CONTRASTS, LABELS)
display(tests.set_index(['subset', 'model_label', 'contrast'])[['n', 'diff', 'p_wilcoxon', 'p_ttest', 'q_bh', 'sig_fdr']].round(4))
for subset, x in tests.groupby('subset', sort=False):
    print(f'{subset:24s} n={int(x.n.iloc[0]):3d}   raw P<0.05 {int(x.sig_raw.sum())}/{len(x)}   after BH-FDR {int(x.sig_fdr.sum())}/{len(x)}')
out.table(tests, 'significance_tests')

## 4. Absence penalty

A graph that holds a drug but has no gene or pathway for it sends the positive to the model blank while the distractors it does hold carry dossiers. This splits every KG arm's pairs by whether the positive's dossier was shown (`pos_has_evidence`, recorded on the row) and gives the paired lift over `none` for each half. The absent half is what a coverage gap costs at deployment; a negative lift means the graph ranked its own blind spot below chance.


In [ ]:
penalty = absence_penalty_table(main, N_BOOT, RNG, LABELS)
display(penalty.set_index(['model_label', 'arm_label', 'positive'])[['n_pairs', 'mrr', 'mrr_none', 'lift', 'lift_lo', 'lift_hi']].round(3))
absent = penalty[penalty.positive == 'absent']
print(f"absent positives: {int(absent.n_pairs.sum())} (model, graph, pair) units; "
      f"mean lift {absent.lift.mean():+.3f} against {penalty[penalty.positive == 'shown'].lift.mean():+.3f} when shown")
out.table(penalty, 'absence_penalty')


## 5. How the models used the evidence

The rank the positive received, and what the model said it relied on.

In [ ]:
out.figure(fig_rank_distribution(main, MODELS, ARMS, LABELS), 'G03b_rank_distribution');
out.figure(fig_evidence_quality(main), 'G03c_evidence_use');

### Visualisation

In [ ]:
out.figure(fig_grounding_gain(arms_main, con_main, con_cov, MODELS, KGS,
                              n_pairs=int(arms_main.n_units.iloc[0]), n_cov_pairs=len(COMMON_PAIRS)),
           'G03a_grounding_gain');

### Checkpoint

In [ ]:
out.checkpoint({
    'chance_mrr': CHANCE, 'n_pairs': int(arms_main.n_units.iloc[0]), 'n_common': len(COMMON_PAIRS),
    'arms_main': arms_main, 'contrasts_main': con_main, 'arms_covered': arms_cov, 'contrasts_covered': con_cov,
    'arms_common': arms_common, 'contrasts_common': con_common, 'tests': tests, 'diagnostics': diag,
    'absence_penalty': penalty,
    'mode': run_ckpt['mode'], 'source_sha256': run_ckpt['source_sha256'],
})

In [ ]:
# Memory cleanup: free state before the next notebook in the pipeline.
for _v in ('kgs', 'graphs', 'dossiers', 'runs', 'bench'):
    globals().pop(_v, None)
gc.collect()
print('done:', out.notebook)